# 真实 0 与自然边界：Two-part、Hurdle、Zero-inflated 与 Fractional response

> 本讲义用于《金融数据分析与建模》课程。配套文件 `05_truncated_twopart_codes_v3.ipynb` 用于生成 `./figs/` 中的配图和 `./data/limitdep_credit_alt_sim.csv`。

本章继续使用企业信贷背景，但不再把贷款金额中的 0 都理解为 Tobit 式的边界归并。很多金融变量中的 0 是真实经济选择。例如企业没有获得贷款、没有发行债券、没有产生贷款笔数，往往不是“潜在负值被压到 0”，而是企业没有参与某个市场或没有跨过某个门槛。

本章的主线是：当 0 是真实选择，或者因变量是比例、计数等自然受限变量时，如何设定比 Tobit 更灵活的模型。

## 1. 从模型地图开始

受限因变量模型的选择，关键不在于因变量表面上是否有很多 0，而在于 0、边界值或 missing 是如何产生的。

![受限因变量模型选择地图](./figs/limit_dep_alt_fig01_model_map.png)

图中灰色虚线框是下一章 Heckman selection 的入口。本章只把它放在地图中，正文不展开。这样处理的原因是：Two-part model 中未参与者的结果通常是 0，而 Heckman selection 中未被选择者的结果变量是不可观测的 missing。

## 2. 为什么贷款金额可以不是 Tobit？

考虑企业银行贷款金额 $B_i$。在数据中，我们经常看到：

$$
B_i=0
$$

一种解释是 Tobit：企业存在潜在净借款需求 $B_i^*$，当 $B_i^*\le 0$ 时，观测贷款金额被归并为 0。此时是否贷款与贷多少来自同一个潜在连续机制。

但另一种解释也很自然：企业是否获得贷款与获得贷款后贷多少，是两个不同的问题。企业可能先要进入银行信贷关系，满足授信条件、抵押要求和审批流程；只有获得贷款之后，才会出现贷款金额的大小问题。

这时，$B_i=0$ 不是潜在贷款金额被归并，而是真实的无贷款状态。这个问题更适合 Two-part model。

## 3. Two-part model：具体的数据生成过程

为了避免泛泛而谈，本章固定使用一个简化但具体的变量设定。

设：

$$
D_i=1(B_i>0)
$$

表示企业是否获得银行贷款。

第一部分解释是否获得贷款：

$$
P(D_i=1\mid z_i)
=
F(z_i'\gamma)
$$

其中：

$$
z_i=(1,\ collateral_i,\ bank\_access_i)
$$

第二部分解释获得贷款后贷多少：

$$
E(B_i\mid B_i>0,x_i)
=
\exp(x_i'\beta)
$$

其中：

$$
x_i=(1,\ collateral_i,\ opportunity_i)
$$

变量设定如下。

| 变量 | 是否进入第一部分 $z_i$ | 是否进入第二部分 $x_i$ | 经济含义 |
|---|---:|---:|---|
| `collateral` | 是 | 是 | 抵押能力既影响贷款可得性，也影响获贷后的授信额度 |
| `bank_access` | 是 | 否 | 银行服务可得性主要影响企业能否进入信贷关系 |
| `opportunity` | 否 | 是 | 投资机会主要影响企业获贷后愿意借多少 |

这个设定的好处是非常清楚：`collateral` 是共同变量，`bank_access` 只影响概率渠道，`opportunity` 只影响强度渠道。

![Two-part model 的概率渠道与强度渠道](./figs/limit_dep_alt_fig02_two_part_mechanism.png)

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv("./data/limitdep_credit_alt_sim.csv")
df[["loan_amt", "D", "collateral", "bank_access", "opportunity"]].describe().round(3)

## 4. Two-part model 的系数含义

第一部分可以使用 Logit 或 Probit。例如使用 Logit：

$$
P(D_i=1\mid z_i)
=
\Lambda(z_i'\gamma)
=
\frac{\exp(z_i'\gamma)}{1+\exp(z_i'\gamma)}
$$

其中，$\gamma_j$ 描述变量如何影响获得贷款的 log odds。它不是对贷款金额的直接边际效应。

第二部分只在 $B_i>0$ 的样本上估计。若使用 log link：

$$
E(B_i\mid B_i>0,x_i)
=
\exp(x_i'\beta)
$$

则 $\beta_j$ 描述变量如何影响正贷款金额的条件均值。它也不是对全样本贷款金额的直接边际效应。

Two-part model 的解释对象不是单独某个方程，而是二者合成后的非条件期望。

## 5. 非条件期望与边际效应

设：

$$
p_i=P(D_i=1\mid z_i)
$$

$$
\mu_i^+=E(B_i\mid B_i>0,x_i)
$$

则非条件期望为：

$$
E(B_i\mid z_i,x_i)
=
p_i\mu_i^+
$$

如果第一部分使用 Logit，第二部分使用 log link，则：

$$
E(B_i\mid z_i,x_i)
=
\Lambda(z_i'\gamma)\exp(x_i'\beta)
$$

对于同时进入两个方程的变量 $w_i$，例如 `collateral`，其边际效应为：

$$
\frac{\partial E(B_i\mid z_i,x_i)}{\partial w_i}
=
\Lambda(z_i'\gamma)
\{1-\Lambda(z_i'\gamma)\}
\gamma_w\mu_i^+
+
p_i\mu_i^+\beta_w
$$

其中：

- 第一项是 **概率渠道**：变量改变企业获得贷款的概率；
- 第二项是 **强度渠道**：变量改变获得贷款后的贷款金额；
- 总边际效应是两个渠道之和。

如果变量只进入第一部分，例如 `bank_access`，则：

$$
\frac{\partial E(B_i\mid z_i,x_i)}{\partial bank\_access_i}
=
p_i(1-p_i)\gamma_{bank\_access}\mu_i^+
$$

如果变量只进入第二部分，例如 `opportunity`，则：

$$
\frac{\partial E(B_i\mid z_i,x_i)}{\partial opportunity_i}
=
p_i\mu_i^+\beta_{opportunity}
$$

In [ ]:
# 读取 codes notebook 生成的 Two-part 平均边际效应
ame = pd.read_csv("./data/two_part_ame.csv")
ame.round(4)

## 6. 结果解释模板

对 `collateral` 的解释应同时包含两个渠道：

> `collateral` 同时进入获得贷款方程和正贷款金额方程。它既可能提高企业获得银行贷款的概率，也可能提高企业获得贷款后的授信额度。因此，其对全样本贷款金额的总影响应分解为概率渠道和强度渠道。

对 `bank_access` 的解释应集中在概率渠道：

> `bank_access` 只进入第一部分。它主要改变企业能否进入银行信贷关系，因此它对非条件贷款金额的影响来自获得贷款概率的变化。

对 `opportunity` 的解释应集中在强度渠道：

> `opportunity` 只进入第二部分。它不直接改变企业是否获得贷款，而是影响已经获得贷款的企业愿意借多少或需要借多少。

## 7. Two-part model 与 Tobit 的差别

![Tobit 与 Two-part 的机制对比](./figs/limit_dep_alt_fig03_tobit_vs_twopart.png)

Tobit 和 Two-part model 可以处理表面上相似的数据，但背后的假设不同。

| 问题 | Tobit | Two-part model |
|---|---|---|
| 0 的含义 | 潜在净借款需求被归并到 0 | 真实的无贷款状态 |
| 机制数量 | 一个潜在连续机制 | 是否贷款与贷多少两个机制 |
| 变量设置 | 同一组 $x$ 同时影响两个方面 | $z_i$ 与 $x_i$ 可以不同 |
| 结果解释 | 系数通过潜变量影响概率和金额 | 非条件期望为 $p_i\mu_i^+$ |

实证中不能只根据 0 的比例来选模型。关键是研究者是否愿意相信 0 和正值由同一个潜在连续机制生成。

::: {.callout-note}
### PE 类产品为什么更接近 Two-part 或 Hurdle？

某些 PE 类理财产品有合格投资者条件和最低投资额。未购买者并不是因为潜在投资额被压到 0，而是可能没有资格、没有参与，或者没有跨过最低投资门槛。因此，这类问题通常更接近 Two-part 或 Hurdle 机制，而不是标准 Tobit。
:::

## 8. Hurdle model：先跨过门槛，再决定正计数

如果结果变量是计数，例如企业当年的贷款笔数、债券发行次数或绿色专利数量，大量 0 也很常见。Hurdle model 的思想是：个体必须先跨过 0 的门槛，之后正计数才由另一个分布决定。

设 $p_i=P(Y_i>0\mid z_i)$，正计数部分服从零截断计数分布，则：

$$
P(Y_i=0\mid z_i)=1-p_i
$$

$$
P(Y_i=k\mid z_i,x_i)
=
p_i
\cdot
\frac{f(k\mid \lambda_i)}
{1-f(0\mid \lambda_i)},
\quad k=1,2,\ldots
$$

在企业信贷背景下，可以把它理解为：企业先决定是否跨过某类贷款或债券融资的最低经济规模；一旦跨过门槛，才产生正的贷款笔数或发行次数。

![Hurdle model 的计数机制](./figs/limit_dep_alt_fig04_hurdle_mechanism.png)

## 9. Zero-inflated model：0 有两个来源

Zero-inflated model 与 Hurdle model 的区别在于：0 可能来自两个来源。

设 $\pi_i$ 是结构性 0 的概率，$f(k\mid \lambda_i)$ 是普通计数分布，则：

$$
P(Y_i=0\mid z_i,x_i)
=
\pi_i+(1-\pi_i)f(0\mid \lambda_i)
$$

$$
P(Y_i=k\mid z_i,x_i)
=
(1-\pi_i)f(k\mid \lambda_i),
\quad k=1,2,\ldots
$$

在贷款笔数例子中，结构性 0 可以理解为企业不具备贷款需求或贷款资格；随机 0 则表示企业具备产生贷款笔数的可能性，但当期恰好没有贷款。

![Zero-inflated model 的两类零值](./figs/limit_dep_alt_fig05_zero_inflated.png)

## 10. Fractional response model：比例因变量的自然边界

有些因变量不是金额或计数，而是比例。例如：

- 银行贷款占总负债比例；
- 短期贷款占银行贷款比例；
- 绿色贷款占总贷款比例；
- 供应链金融融资占总融资比例。

这类变量满足：

$$
0\le s_i\le 1
$$

如果直接用 OLS：

$$
s_i=x_i'\beta+\varepsilon_i
$$

预测值可能小于 0 或大于 1。Fractional response model 直接设定条件均值在 $[0,1]$ 内：

$$
E(s_i\mid x_i)=G(x_i'\beta)
$$

常用的 $G(\cdot)$ 是 logistic 函数：

$$
G(a)=\frac{\exp(a)}{1+\exp(a)}
$$

边际效应为：

$$
\frac{\partial E(s_i\mid x_i)}{\partial x_{ij}}
=
G(x_i'\beta)\{1-G(x_i'\beta)\}\beta_j
$$

平均边际效应为：

$$
AME_j
=
\frac{1}{n}
\sum_{i=1}^n
G(x_i'\hat\beta)\{1-G(x_i'\hat\beta)\}\hat\beta_j
$$

![FRM 与 OLS 的预测对比](./figs/limit_dep_alt_fig06_frm_vs_ols.png)

图中 OLS 预测值可能接近或越过自然边界，而 Fractional response model 的预测值天然落在 $[0,1]$。因此，在比例因变量中，FRM 不只是“非线性版本的 OLS”，而是把因变量的自然边界直接写进了条件均值。

## 11. 小结

本章的核心是识别 0 的经济含义。

| 模型 | 关键问题 | 信贷例子 |
|---|---|---|
| Two-part model | 是否发生与发生多少是否分开 | 是否贷款与贷款金额 |
| Hurdle model | 是否跨过 0 的门槛 | 是否达到贷款或债券融资的最低经济规模 |
| Zero-inflated model | 0 是否有两个来源 | 贷款笔数中的结构性 0 与随机 0 |
| Fractional response model | 因变量是否是比例 | 银行贷款占总负债比例 |

在实证研究中，同一个贷款金额变量可以被不同模型解释。关键不是模型名称，而是研究者对数据生成机制的判断是否清楚、具体、可辩护。